## Step 1: Load Dataset and Inspect Schema

Before writing any code we need to understand the raw structure of the data.
We need to know: what are the column names, what type is each column, and what does a real sample look like.
This tells us exactly what fields we can use and what we need to transform.

In [10]:
from datasets import load_dataset
from transformers import AutoTokenizer
import collections
import pandas as pd
import numpy as np
import torch

In [2]:
dataset = load_dataset("yqzheng/semeval2014_restaurants")

In [10]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'aspect', 'start', 'end', 'label'],
        num_rows: 3608
    })
    test: Dataset({
        features: ['text', 'aspect', 'start', 'end', 'label'],
        num_rows: 1120
    })
})


In [9]:
print(dataset["train"][:1])

{'text': ['But the staff was so horrible to us.'], 'aspect': ['staff'], 'start': [8], 'end': [13], 'label': [-1]}


## Step 2: Inspect a Real Sample

We need to see what an actual row looks like before assuming anything about the data format.
Key questions:
- What does the text look like?
- Is the aspect a single word or a phrase?
- What is the label value — is it 0/1/2 or something else?

In [11]:
print("Train sample 1:", dataset["train"][0])
print("Train sample 2:", dataset["train"][1])
print("Train sample 3:", dataset["train"][2])
print()
print("Features:", dataset["train"].features)

Train sample 1: {'text': 'But the staff was so horrible to us.', 'aspect': 'staff', 'start': 8, 'end': 13, 'label': -1}
Train sample 2: {'text': "To be completely fair, the only redeeming factor was the food, which was above average, but couldn't make up for all the other deficiencies of Teodora.", 'aspect': 'food', 'start': 57, 'end': 61, 'label': 1}
Train sample 3: {'text': "The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not.", 'aspect': 'food', 'start': 4, 'end': 8, 'label': 1}

Features: {'text': Value('string'), 'aspect': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('int64')}


## Step 3: Check Label Distribution

This is critical for two reasons:

**1. Label remapping** — PyTorch CrossEntropyLoss requires class indices starting from 0.
If labels are -1, 0, 1 we must remap them to 0, 1, 2 before training or the loss function will crash.

**2. Class imbalance** — If one class (e.g. positive) dominates the dataset,
the model will learn to always predict that class and still achieve high accuracy
while being completely useless for minority classes.
If imbalance is severe we need weighted loss to penalise the majority class.

In [5]:
train_labels = collections.Counter(dataset["train"]["label"])
test_labels  = collections.Counter(dataset["test"]["label"])

print("Train label distribution:", dict(sorted(train_labels.items())))
print("Test  label distribution:", dict(sorted(test_labels.items())))

Train label distribution: {-1: 807, 0: 637, 1: 2164}
Test  label distribution: {-1: 196, 0: 196, 1: 728}


In [ ]:
df = pd.DataFrame({
    "label_raw": sorted(train_labels.keys()),
    "train_count": [train_labels[k] for k in sorted(train_labels.keys())],
    "test_count":  [test_labels[k]  for k in sorted(test_labels.keys())],
})
df["train_%"] = (df["train_count"] / df["train_count"].sum() * 100).round(1)
df["test_%"]  = (df["test_count"]  / df["test_count"].sum() * 100).round(1)
print()
print(df.to_string(index=False))


 label_raw  train_count  test_count  train_%  test_%
        -1          807         196     22.4    17.5
         0          637         196     17.7    17.5
         1         2164         728     60.0    65.0


## Step 4: Decide on Label Remapping

From the distribution above we can see the raw label values.
PyTorch requires labels to be 0-indexed integers.

Decision:
- Raw label -1 → 0 (negative)
- Raw label  0 → 1 (neutral)
- Raw label  1 → 2 (positive)

We also assess class imbalance here. If any class represents less than 15% of the data
we will use weighted CrossEntropyLoss during training.

In [7]:
# Define the remapping based on what we observed above
LABEL_REMAP = {-1: 0, 0: 1, 1: 2}
ID2LABEL    = {0: "negative", 1: "neutral", 2: "positive"}

# Check imbalance ratio
total = sum(train_labels.values())
for raw_label, remapped in LABEL_REMAP.items():
    count = train_labels[raw_label]
    pct   = count / total * 100
    print(f"  raw={raw_label:>2} → remapped={remapped} ({ID2LABEL[remapped]:>8}) | count={count:>4} | {pct:.1f}%")

# Imbalance decision
max_count = max(train_labels.values())
min_count = min(train_labels.values())
ratio = max_count / min_count
print(f"\nImbalance ratio (max/min): {ratio:.1f}x")
if ratio > 3:
    print("→ DECISION: Use weighted CrossEntropyLoss — dataset is imbalanced")
else:
    print("→ DECISION: Standard CrossEntropyLoss is sufficient")

  raw=-1 → remapped=0 (negative) | count= 807 | 22.4%
  raw= 0 → remapped=1 ( neutral) | count= 637 | 17.7%
  raw= 1 → remapped=2 (positive) | count=2164 | 60.0%

Imbalance ratio (max/min): 3.4x
→ DECISION: Use weighted CrossEntropyLoss — dataset is imbalanced


## Step 5: Validate the Input Format

The core idea of ABSA is to condition the model on both the text AND the aspect.
We use the format:
aspect: {aspect} [SEP] {text}
The `[SEP]` token is DeBERTa's segment separator. It signals to the model that
what comes before is the query (which aspect are we asking about?)
and what comes after is the context (the review text).

Without this, the model scores general sentence sentiment — exactly the problem we are fixing.

Here we verify this format looks correct on real samples before committing to it.

In [8]:
def build_absa_input(text: str, aspect: str) -> str:
    return f"aspect: {aspect} [SEP] {text}"

# Test on real samples from the dataset
print("=== Formatted ABSA Inputs ===\n")
for i in range(5):
    sample = dataset["train"][i]
    formatted = build_absa_input(sample["text"], sample["aspect"])
    raw_label  = sample["label"]
    remapped   = LABEL_REMAP[raw_label]
    sentiment  = ID2LABEL[remapped]
    print(f"Input:     {formatted}")
    print(f"Label:     {sentiment} (raw={raw_label} → remapped={remapped})")
    print()

=== Formatted ABSA Inputs ===

Input:     aspect: staff [SEP] But the staff was so horrible to us.
Label:     negative (raw=-1 → remapped=0)

Input:     aspect: food [SEP] To be completely fair, the only redeeming factor was the food, which was above average, but couldn't make up for all the other deficiencies of Teodora.
Label:     positive (raw=1 → remapped=2)

Input:     aspect: food [SEP] The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not.
Label:     positive (raw=1 → remapped=2)

Input:     aspect: kitchen [SEP] The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or not.
Label:     positive (raw=1 → remapped=2)

Input:     aspect: menu [SEP] The food is uniformly exceptional, with a very capable kitchen which will proudly whip up whatever you feel like eating, whether it's on the menu or 

## Step 6: Check Token Length Distribution

DeBERTa has a maximum input length of 512 tokens.
We need to verify our formatted inputs (`aspect: X [SEP] text`) fit within a safe
truncation limit so we don't silently cut off important content.

If 95%+ of inputs are under 128 tokens we can use max_length=128 which:
- Speeds up training significantly
- Reduces memory usage
- Still captures the full input for nearly all samples

In [9]:
print("Loading DeBERTa tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

lengths = []
for sample in dataset["train"]:
    formatted = build_absa_input(sample["text"], sample["aspect"])
    tokens    = tokenizer(formatted, truncation=False)
    lengths.append(len(tokens["input_ids"]))

lengths = np.array(lengths)
print(f"Min tokens:    {lengths.min()}")
print(f"Max tokens:    {lengths.max()}")
print(f"Mean tokens:   {lengths.mean():.1f}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")
print(f"99th percentile: {np.percentile(lengths, 99):.0f}")
print()
pct_under_128 = (lengths <= 128).mean() * 100
print(f"% of samples under 128 tokens: {pct_under_128:.1f}%")
print()
if pct_under_128 >= 95:
    print("→ DECISION: max_length=128 is safe — covers 95%+ of all samples")
else:
    print("→ DECISION: max_length=128 will truncate too many samples — consider 256")

Loading DeBERTa tokenizer...
Min tokens:    9
Max tokens:    89
Mean tokens:   27.7
95th percentile: 48
99th percentile: 62

% of samples under 128 tokens: 100.0%

→ DECISION: max_length=128 is safe — covers 95%+ of all samples


## Step 7: Compute Class Weights for Weighted Loss

If the dataset is imbalanced (as detected in Step 4), we compute class weights
to pass into CrossEntropyLoss during training.

The formula is: weight[i] = total_samples / (num_classes * count[i])

This means minority classes get a higher weight — the model is penalised more
for getting them wrong, which forces it to learn them properly rather than
ignoring them in favour of the majority class.

In [11]:
remapped_counts = {
    LABEL_REMAP[raw]: count
    for raw, count in train_labels.items()
}

total_samples = sum(remapped_counts.values())
num_classes   = 3

weights = []
for class_idx in range(num_classes):
    w = total_samples / (num_classes * remapped_counts[class_idx])
    weights.append(round(w, 4))
    print(f"  {ID2LABEL[class_idx]:>8} (class {class_idx}): count={remapped_counts[class_idx]:>4} → weight={w:.4f}")

class_weights = torch.tensor(weights, dtype=torch.float)
print(f"\nFinal class weights tensor: {class_weights}")
print("→ These will be passed to CrossEntropyLoss(weight=class_weights) in train_absa.py")

  negative (class 0): count= 807 → weight=1.4903
   neutral (class 1): count= 637 → weight=1.8880
  positive (class 2): count=2164 → weight=0.5558

Final class weights tensor: tensor([1.4903, 1.8880, 0.5558])
→ These will be passed to CrossEntropyLoss(weight=class_weights) in train_absa.py


## Step 8: Summary of All Decisions

This cell consolidates every decision made in this notebook.
These decisions are directly encoded in the training code — nothing is assumed or guessed.

In [12]:
print("=" * 55)
print("DECISIONS MADE — DEBERTA ABSA TRAINING CONFIG")
print("=" * 55)
print()
print("Dataset:       yqzheng/semeval2014_restaurants")
print(f"Train size:    {len(dataset['train'])}")
print(f"Test size:     {len(dataset['test'])}")
print()
print("Label remapping:")
for raw, remapped in LABEL_REMAP.items():
    print(f"  {raw:>2} → {remapped} ({ID2LABEL[remapped]})")
print()
print(f"Input format:  'aspect: {{aspect}} [SEP] {{text}}'")
print(f"Max length:    128 tokens")
print(f"Model:         microsoft/deberta-v3-base")
print(f"Num labels:    3 (negative / neutral / positive)")
print()
print(f"Class weights: {class_weights.tolist()}")
print(f"Loss function: CrossEntropyLoss(weight=class_weights)")
print()
print("These values are hardcoded into:")
print("  → deberta_absa/data_preprocessing.py")
print("  → deberta_absa/train_absa.py")

DECISIONS MADE — DEBERTA ABSA TRAINING CONFIG

Dataset:       yqzheng/semeval2014_restaurants
Train size:    3608
Test size:     1120

Label remapping:
  -1 → 0 (negative)
   0 → 1 (neutral)
   1 → 2 (positive)

Input format:  'aspect: {aspect} [SEP] {text}'
Max length:    128 tokens
Model:         microsoft/deberta-v3-base
Num labels:    3 (negative / neutral / positive)

Class weights: [1.4903000593185425, 1.8880000114440918, 0.5558000206947327]
Loss function: CrossEntropyLoss(weight=class_weights)

These values are hardcoded into:
  → deberta_absa/data_preprocessing.py
  → deberta_absa/train_absa.py
